# k-Nearest Neighbors (kNN) nel lavoro di uno sviluppatore software

**Obiettivo del notebook**: vedere, con due casi concreti tratti dalla vita di un team di sviluppo, perché l'algoritmo **kNN** è uno strumento utile e facile da spiegare.

| Parte | Tipo di problema | Domanda a cui rispondiamo |
|---|---|---|
| **A** | Classificazione | *Questa build della pipeline CI fallirà?* |
| **B** | Regressione | *Quante ore servirà per risolvere questo ticket?* |

I dati sono **sintetici** (generati dentro il notebook con un seme fisso), quindi il notebook è autosufficiente: non servono file esterni. I due dataset vengono anche salvati in CSV, così possiamo aprirli con altri strumenti.

---

## Glossario minimo (lo riprendiamo man mano)

- **Feature** (o variabile esplicativa): una colonna che descrive un esempio, ad es. il numero di righe modificate in una build.
- **Target** (o etichetta): ciò che vogliamo prevedere, ad es. "build fallita sì/no".
- **Classificazione**: il target è una categoria (fallita / riuscita).
- **Regressione**: il target è un numero continuo (ore di lavoro).
- **Vicino** (*neighbor*): un esempio del passato "simile" a quello nuovo, dove la somiglianza è misurata da una **distanza** tra i vettori di feature.
- **k**: quanti vicini consideriamo.
- **Distanza euclidea**: la distanza "in linea d'aria" tra due punti; con feature $x$ e $y$ vale $\sqrt{\sum_i (x_i - y_i)^2}$.

## L'idea del kNN in una frase

> Per prevedere un caso nuovo, cerchiamo i **k casi passati più simili** e ci fidiamo di loro:
> - in **classificazione** facciamo votare i k vicini (vince la classe più frequente);
> - in **regressione** facciamo la media dei loro valori.

Il kNN è detto **apprendimento "pigro"** (*lazy learning*): in fase di addestramento non costruisce alcuna formula, si limita a **memorizzare** gli esempi. Tutto il lavoro avviene al momento della previsione, quando si calcolano le distanze.

## La teoria del KNN

**La meccanica di base**<br>

![](knn_1.png)

---
![](knn_2.png)

---
![](knn_3.png)

## Spazi multi-dimensionali e distanza euclidea

![](recap_AL_1.png)

[Link Wikipedia](https://it.wikipedia.org/wiki/Distanza_euclidea) alla distanza euclidea (il teorema di Pitagora (500 ac!) applicato a n dimensioni)

---
![](recap_AL_2.png)

---
![](recap_AL_3.png)

## Un famoso esempio

![](knn_6.png)

---
![](knn_7.png)

---
![](knn_8.png)

## Perché interessa a un team di sviluppo

1. **È spiegabile**: possiamo sempre dire *"questa previsione nasce da questi 5 casi del passato"*, e mostrarli. In un team questo convince più di un coefficiente.
2. **Si adatta a relazioni non lineari** senza doverle scrivere a mano.
3. **È la stessa idea della ricerca per similarità**: trovare ticket duplicati, suggerire il revisore che ha lavorato su codice simile, e (con gli *embedding*, cioè vettori numerici che rappresentano testi) la ricerca semantica usata nei sistemi RAG.

## 0. Preparazione dell'ambiente

Librerie necessarie (se mancano, le installiamo nell'ambiente conda attivo):

```
pip install numpy pandas matplotlib scikit-learn
```

- **numpy / pandas**: calcolo numerico e tabelle.
- **matplotlib**: grafici.
- **scikit-learn** (abbreviato `sklearn`): la libreria di machine learning che contiene il kNN.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
)

# Seme casuale fisso: rende i risultati riproducibili a ogni esecuzione
SEED = 42
rng = np.random.default_rng(SEED)

pd.set_option("display.float_format", "{:.2f}".format)
plt.rcParams["figure.figsize"] = (8, 4.5)
print("Ambiente pronto")

---
# PARTE A — Classificazione: *questa build CI fallirà?*

## A.1 Il problema

La **pipeline CI** (*Continuous Integration*, integrazione continua) compila e testa automaticamente il codice a ogni push. Una build fallita blocca il team e costa tempo.

Se potessimo stimare **prima di lanciarla** la probabilità che una build fallisca, potremmo per esempio:
- chiedere una revisione più attenta per le modifiche "a rischio";
- dare priorità alle build rischiose nella coda;
- far girare prima la suite di test più lenta solo quando serve.

## A.2 Il dataset

Ogni riga è una build passata. Le feature sono:

| Colonna | Significato |
|---|---|
| `righe_modificate` | righe di codice aggiunte + rimosse nel commit |
| `file_modificati` | numero di file toccati |
| `test_modificati` | numero di file di test aggiunti o modificati |
| `dipendenze_aggiornate` | 1 se il commit aggiorna librerie esterne, 0 altrimenti |
| `ore_da_ultima_build_verde` | ore trascorse dall'ultima build riuscita sul ramo |
| `commit_autore_nel_repo` | quanti commit ha già fatto l'autore su questo repository (proxy dell'esperienza) |
| `tipo_ramo` | `feature`, `hotfix` o `release` (variabile **categorica**, cioè a valori testuali) |
| **`fallita`** | **target**: 1 = build fallita, 0 = build riuscita |

Generiamo i dati in modo che il rischio di fallimento cresca con le modifiche grandi, le dipendenze aggiornate e il tempo trascorso dall'ultima build verde, e diminuisca con l'esperienza e con i test scritti.

In [ ]:
def genera_build_ci(n=1500, rng=rng):
    righe = np.round(rng.lognormal(mean=4.5, sigma=1.1, size=n)).clip(1, 6000)
    file_mod = np.round(1 + righe / 60 + rng.gamma(2, 2, n)).clip(1, 120)
    test_mod = rng.poisson(1.5 + file_mod / 12)
    dip = rng.binomial(1, 0.15, n)
    ore_verde = rng.exponential(12, n).clip(0, 120)
    esperienza = np.round(rng.gamma(1.5, 60, n)).clip(0, 800)
    ramo = rng.choice(["feature", "hotfix", "release"], size=n, p=[0.7, 0.2, 0.1])

    # "Punteggio di rischio" latente (non osservato): la verità che il modello deve scoprire
    rischio = (
        -1.2
        + 1.4 * np.log1p(righe) / np.log(1000)          # modifiche grandi = più rischio
        + 1.6 * dip                                      # aggiornare librerie è rischioso
        + 0.9 * (ore_verde > 24)                         # ramo "rimasto rosso/fermo" a lungo
        - 1.0 * np.tanh(esperienza / 150)               # l'esperienza protegge
        - 0.25 * np.minimum(test_mod, 6) / 2             # scrivere test protegge un po'
        + np.where(ramo == "hotfix", 0.6, 0.0)           # hotfix fatti di fretta
        + np.where(ramo == "release", -0.4, 0.0)         # release più controllate
    )
    p_fail = 1 / (1 + np.exp(-2.5 * rischio))
    fallita = rng.binomial(1, p_fail)

    return pd.DataFrame({
        "righe_modificate": righe.astype(int),
        "file_modificati": file_mod.astype(int),
        "test_modificati": test_mod.astype(int),
        "dipendenze_aggiornate": dip,
        "ore_da_ultima_build_verde": ore_verde.round(1),
        "commit_autore_nel_repo": esperienza.astype(int),
        "tipo_ramo": ramo,
        "fallita": fallita,
    })

build = genera_build_ci()
build.to_csv("build_ci_classificazione.csv", index=False)
print(build.shape)
build.head()

## A.3 Analisi esplorativa

Prima di qualunque modello guardiamo i dati. Tre domande:
1. Le classi sono **bilanciate**? (cioè: le build fallite sono circa quante le riuscite?)
2. Le feature hanno **scale** molto diverse? (Per il kNN è decisivo: lo vediamo tra poco.)
3. Le feature sembrano davvero legate al fallimento?

In [ ]:
# 1) Bilanciamento delle classi
print("Distribuzione del target (proporzioni):")
print(build["fallita"].value_counts(normalize=True).rename({0: "riuscita", 1: "fallita"}))

In [ ]:
# 2) Scale delle feature numeriche: minimo, massimo, deviazione standard
num_cols_A = ["righe_modificate", "file_modificati", "test_modificati",
              "dipendenze_aggiornate", "ore_da_ultima_build_verde", "commit_autore_nel_repo"]
build[num_cols_A].describe().T[["min", "max", "mean", "std"]]

**Osservazione chiave**: `righe_modificate` arriva a migliaia, `dipendenze_aggiornate` vale solo 0 o 1. Se calcoliamo la distanza euclidea sui valori grezzi, una differenza di 200 righe "pesa" 200 volte più di una differenza sulle dipendenze: in pratica il kNN guarderebbe **quasi soltanto** le righe modificate. Teniamolo a mente.

In [ ]:
# 3) Tasso di fallimento per alcune feature
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

build.groupby("tipo_ramo")["fallita"].mean().plot.bar(ax=axes[0], color="#1b9e77")
axes[0].set_title("Tasso di fallimento per tipo di ramo"); axes[0].set_ylabel("quota fallite")

build.groupby("dipendenze_aggiornate")["fallita"].mean().plot.bar(ax=axes[1], color="#7570b3")
axes[1].set_title("... per dipendenze aggiornate (0/1)")

fasce = pd.qcut(build["righe_modificate"], 5)
build.groupby(fasce, observed=True)["fallita"].mean().plot.bar(ax=axes[2], color="#d95f02")
axes[2].set_title("... per quintile di righe modificate")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()

## A.4 Divisione in training set e test set

- **Training set** (80%): gli esempi che il kNN "memorizza".
- **Test set** (20%): esempi tenuti da parte, mai visti, su cui misuriamo onestamente le prestazioni.

Usiamo `stratify=y`: la **stratificazione** mantiene la stessa proporzione di build fallite nei due insiemi.

In [ ]:
X_A = build.drop(columns="fallita")
y_A = build["fallita"]

X_A_train, X_A_test, y_A_train, y_A_test = train_test_split(
    X_A, y_A, test_size=0.2, stratify=y_A, random_state=SEED
)
print("Training:", X_A_train.shape, " Test:", X_A_test.shape)

## A.5 Un riferimento minimo: il modello *baseline*

Un **baseline** è un modello banale che serve da metro di paragone. Il `DummyClassifier` con strategia `most_frequent` risponde sempre con la classe più frequente ("riuscita").

Le metriche che usiamo:
- **Accuracy**: quota di previsioni corrette sul totale.
- **Precision** (sulla classe "fallita"): tra le build che il modello segnala come fallite, quante lo sono davvero.
- **Recall** (sulla classe "fallita"): tra le build davvero fallite, quante il modello ne intercetta.
- **F1**: media armonica di precision e recall; un solo numero che le bilancia.

Con classi sbilanciate l'accuracy da sola inganna: il baseline ha un'accuracy discreta ma recall zero, cioè non trova **nessuna** build fallita.

In [ ]:
def metriche_class(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred),
    }

risultati_A = {}

baseline_A = DummyClassifier(strategy="most_frequent").fit(X_A_train, y_A_train)
risultati_A["baseline"] = metriche_class(y_A_test, baseline_A.predict(X_A_test))
pd.DataFrame(risultati_A).T

## A.6 kNN: prima la versione *as-is*, poi quella *idiomatica*

### Versione as-is (ingenua)

Quello che faremmo di getto: trasformiamo la colonna testuale `tipo_ramo` in numeri con `pd.get_dummies` (**one-hot encoding**: una colonna 0/1 per ogni categoria) e passiamo tutto al kNN **senza standardizzare**. Usiamo k = 5, il valore predefinito di scikit-learn.

In [ ]:
# --- AS-IS: nessuna standardizzazione ---
Xtr_raw = pd.get_dummies(X_A_train, columns=["tipo_ramo"], dtype=int)
Xte_raw = pd.get_dummies(X_A_test, columns=["tipo_ramo"], dtype=int).reindex(columns=Xtr_raw.columns, fill_value=0)

knn_raw = KNeighborsClassifier(n_neighbors=5).fit(Xtr_raw, y_A_train)
risultati_A["kNN as-is (k=5, no scaling)"] = metriche_class(y_A_test, knn_raw.predict(Xte_raw))
pd.DataFrame(risultati_A).T

### Versione idiomatica: `Pipeline` + standardizzazione

La **standardizzazione** (`StandardScaler`) trasforma ogni feature numerica sottraendo la media e dividendo per la deviazione standard: dopo, tutte hanno media 0 e deviazione standard 1, quindi **contano allo stesso modo** nella distanza.

Mettiamo tutto in una **`Pipeline`**, cioè una catena di passi (pre-elaborazione → modello) che si comporta come un unico oggetto. Vantaggi:
- la media e la deviazione standard vengono calcolate **solo sul training set** (evitiamo il *data leakage*, cioè la "fuga" di informazioni dal test set nell'addestramento);
- in produzione basta chiamare `pipeline.predict(nuovi_dati)` sui dati grezzi.

Il **`ColumnTransformer`** applica trasformazioni diverse a colonne diverse: `StandardScaler` alle numeriche, `OneHotEncoder` alla categorica.

In [ ]:
# --- IDIOMATICO: Pipeline con standardizzazione ---
cat_cols_A = ["tipo_ramo"]

preproc_A = ColumnTransformer([
    ("num", StandardScaler(), num_cols_A),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_A),
])

pipe_A = Pipeline([
    ("prep", preproc_A),
    ("knn", KNeighborsClassifier(n_neighbors=5)),
])

pipe_A.fit(X_A_train, y_A_train)
risultati_A["kNN idiomatico (k=5, scaling)"] = metriche_class(y_A_test, pipe_A.predict(X_A_test))
pd.DataFrame(risultati_A).T

**Lezione pratica n. 1**: con il kNN la standardizzazione non è un dettaglio. Senza, il modello misura la somiglianza quasi solo sulle righe modificate e ignora segnali importanti come l'aggiornamento delle dipendenze.

## A.7 Scegliere k con la cross-validazione

**k** è un **iperparametro**: un valore che non viene appreso dai dati ma scelto da noi.
- k piccolo (es. 1): il modello segue ogni singolo caso, anche il rumore → **overfitting** (si adatta troppo al training set e generalizza male).
- k grande: il modello "media" troppo e perde i dettagli → **underfitting**.

Proviamo anche il parametro `weights`:
- `uniform`: ogni vicino vota con lo stesso peso;
- `distance`: i vicini più vicini pesano di più (peso = 1/distanza).

Per scegliere usiamo la **cross-validazione a 5 fold**: il training set viene diviso in 5 parti; a turno si addestra su 4 e si valuta sulla quinta; si fa la media dei 5 punteggi. `GridSearchCV` prova tutte le combinazioni della **griglia** di valori. Ottimizziamo l'**F1**, perché ci interessa trovare le build fallite, non solo avere un'accuracy alta.

In [ ]:
griglia_A = {
    "knn__n_neighbors": list(range(1, 52, 2)),   # k dispari: evitiamo pareggi nel voto
    "knn__weights": ["uniform", "distance"],
}

cv_A = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
gs_A = GridSearchCV(pipe_A, griglia_A, cv=cv_A, scoring="f1", n_jobs=-1)
gs_A.fit(X_A_train, y_A_train)

print("Migliori iperparametri:", gs_A.best_params_)
print(f"F1 medio in cross-validazione: {gs_A.best_score_:.3f}")

In [ ]:
# Curva F1 in funzione di k
cvres = pd.DataFrame(gs_A.cv_results_)
for w, col in [("uniform", "#1b9e77"), ("distance", "#d95f02")]:
    sub = cvres[cvres["param_knn__weights"] == w]
    plt.plot(sub["param_knn__n_neighbors"], sub["mean_test_score"], marker="o", ms=3, label=w, color=col)
plt.xlabel("k (numero di vicini)"); plt.ylabel("F1 medio (CV)")
plt.title("Scelta di k per la classificazione delle build")
plt.legend(title="weights"); plt.grid(alpha=0.3); plt.show()

## A.8 Valutazione finale sul test set

Il `classification_report` mostra precision, recall e F1 per entrambe le classi. La **matrice di confusione** incrocia classe vera (righe) e classe prevista (colonne):
- in basso a destra: build fallite riconosciute (**veri positivi**);
- in basso a sinistra: build fallite sfuggite (**falsi negativi**) — le più costose per noi;
- in alto a destra: falsi allarmi (**falsi positivi**).

In [ ]:
best_A = gs_A.best_estimator_
y_A_pred = best_A.predict(X_A_test)
risultati_A[f"kNN ottimizzato {gs_A.best_params_}"] = metriche_class(y_A_test, y_A_pred)

print(classification_report(y_A_test, y_A_pred, target_names=["riuscita", "fallita"]))
ConfusionMatrixDisplay.from_predictions(y_A_test, y_A_pred, display_labels=["riuscita", "fallita"], cmap="Blues")
plt.title("Matrice di confusione — test set"); plt.show()

pd.DataFrame(risultati_A).T

### Probabilità e soglia di decisione

Il kNN restituisce anche una **probabilità** con `predict_proba`: è la quota (eventualmente pesata) di vicini fallite. Per default si segnala "fallita" se la probabilità supera 0,5, ma questa **soglia** è una scelta di business: se una build fallita costa molto, possiamo abbassarla e accettare più falsi allarmi pur di intercettare più fallimenti.

Nota: con pesi `uniform` e k vicini, la probabilità può assumere solo i valori 0, 1/k, 2/k, …, 1 (per k = 5: 0; 0,2; 0,4; …). Per questo soglie diverse possono dare risultati identici.

In [ ]:
proba_A = best_A.predict_proba(X_A_test)[:, 1]
righe_soglia = []
for soglia in [0.2, 0.3, 0.4, 0.5, 0.6]:
    pred = (proba_A >= soglia).astype(int)
    righe_soglia.append({"soglia": soglia,
                         "precision": precision_score(y_A_test, pred, zero_division=0),
                         "recall": recall_score(y_A_test, pred),
                         "build segnalate": int(pred.sum())})
pd.DataFrame(righe_soglia)

## A.9 Il punto di forza: *spiegare* una previsione mostrando i vicini

Arriva una nuova build. Il modello la giudica a rischio? E soprattutto: **perché?** Con il kNN la risposta è concreta: ecco le build passate più simili e come sono finite.

Per trovarle applichiamo alla nuova build la stessa pre-elaborazione della pipeline (`best_A.named_steps["prep"]`) e poi chiediamo al kNN i vicini con il metodo `kneighbors`, che restituisce distanze e posizioni nel training set.

In [ ]:
nuova_build = pd.DataFrame([{
    "righe_modificate": 600,
    "file_modificati": 15,
    "test_modificati": 0,
    "dipendenze_aggiornate": 1,
    "ore_da_ultima_build_verde": 30.0,
    "commit_autore_nel_repo": 20,
    "tipo_ramo": "hotfix",
}])

p = best_A.predict_proba(nuova_build)[0, 1]
print(f"Probabilità stimata di fallimento: {p:.0%}")

def mostra_vicini(pipeline, X_train, y_train, nuovo, n=5):
    # Restituisce i n casi del training set più simili al nuovo esempio
    X_trasf = pipeline.named_steps["prep"].transform(nuovo)
    dist, idx = pipeline.steps[-1][1].kneighbors(X_trasf, n_neighbors=n)
    vicini = X_train.iloc[idx[0]].copy()
    vicini["target"] = y_train.iloc[idx[0]].values
    vicini["distanza"] = dist[0]
    return vicini

mostra_vicini(best_A, X_A_train, y_A_train, nuova_build, n=best_A.named_steps["knn"].n_neighbors)

Questo è il messaggio che possiamo mettere in un commento automatico sulla pull request:
*"Build simile a N build passate, di cui M fallite: consigliata revisione aggiuntiva."*
Pochi altri algoritmi offrono una spiegazione così immediata.

# A.10 Decision Boundary

Il Decision Boundary

![](knn_4.png)

---
![](knn_5.png)

---


---
# PARTE B — Regressione: *quante ore per risolvere questo ticket?*

## B.1 Il problema

Stimare il tempo di risoluzione dei ticket serve per pianificare lo sprint, rispettare gli **SLA** (*Service Level Agreement*, i tempi di risposta promessi al cliente) e bilanciare il carico tra le persone.

## B.2 Il dataset

| Colonna | Significato |
|---|---|
| `priorita` | da 1 (critica) a 4 (bassa) |
| `componente` | `frontend`, `backend`, `database`, `infrastruttura` (categorica) |
| `servizi_coinvolti` | quanti microservizi sono toccati dal problema |
| `riproducibile` | 1 se il bug è riproducibile in locale, 0 altrimenti |
| `commenti_iniziali` | numero di commenti nelle prime 24 ore (proxy della confusione sul problema) |
| `esperienza_assegnatario` | anni di esperienza di chi prende il ticket |
| **`ore_risoluzione`** | **target**: ore di lavoro effettive |

Nei dati generati inseriamo di proposito relazioni **non lineari** e **interazioni** (un effetto che dipende da un'altra variabile): per esempio, un bug *non riproducibile* sul *database* costa molto più della somma dei due effetti presi separatamente. È il tipo di situazione in cui il kNN mostra il suo valore rispetto a una regressione lineare.

In [ ]:
def genera_ticket(n=1200, rng=rng):
    priorita = rng.choice([1, 2, 3, 4], size=n, p=[0.1, 0.3, 0.4, 0.2])
    componente = rng.choice(["frontend", "backend", "database", "infrastruttura"],
                            size=n, p=[0.35, 0.35, 0.15, 0.15])
    servizi = 1 + rng.poisson(1.2, n)
    riproducibile = rng.binomial(1, 0.65, n)
    commenti = rng.poisson(2 + 1.5 * (1 - riproducibile), n)
    esperienza = rng.gamma(2.5, 2.2, n).clip(0.5, 25).round(1)

    base = {"frontend": 3, "backend": 5, "database": 7, "infrastruttura": 8}
    ore = (
        np.vectorize(base.get)(componente)
        * (1 + 0.6 * (servizi - 1) ** 1.3)                      # crescita più che lineare con i servizi
        * np.where(riproducibile == 1, 1.0,
                   np.where(np.isin(componente, ["database", "infrastruttura"]), 3.0, 1.8))  # interazione
        * (1.6 / (1 + np.log1p(esperienza)))                    # rendimenti decrescenti dell'esperienza
        * (1 + 0.08 * commenti)
        * np.where(priorita == 1, 0.8, 1.0)                     # i critici vengono "attaccati" subito
    )
    ore = ore * rng.lognormal(0, 0.25, n)                       # rumore moltiplicativo
    return pd.DataFrame({
        "priorita": priorita,
        "componente": componente,
        "servizi_coinvolti": servizi,
        "riproducibile": riproducibile,
        "commenti_iniziali": commenti,
        "esperienza_assegnatario": esperienza,
        "ore_risoluzione": ore.round(1),
    })

ticket = genera_ticket()
ticket.to_csv("ticket_risoluzione_knn.csv", index=False)
print(ticket.shape)
ticket.head()

## B.3 Analisi esplorativa

In [ ]:
ticket.describe().T[["min", "max", "mean", "50%", "std"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(ticket["ore_risoluzione"], bins=40, color="#7570b3")
axes[0].set_title("Distribuzione delle ore di risoluzione"); axes[0].set_xlabel("ore")

ticket.groupby(["componente", "riproducibile"])["ore_risoluzione"].median().unstack().plot.bar(
    ax=axes[1], color=["#d95f02", "#1b9e77"])
axes[1].set_title("Ore mediane: componente × riproducibile")
axes[1].legend(["non riproducibile", "riproducibile"]); axes[1].tick_params(axis="x", rotation=30)

axes[2].scatter(ticket["esperienza_assegnatario"], ticket["ore_risoluzione"], s=8, alpha=0.4, color="#1b9e77")
axes[2].set_title("Ore vs esperienza dell'assegnatario"); axes[2].set_xlabel("anni")

plt.tight_layout(); plt.show()

Il grafico centrale mostra l'**interazione**: essere non riproducibile allunga i tempi soprattutto su database e infrastruttura. Il grafico a destra mostra un effetto **non lineare**: i primi anni di esperienza riducono molto i tempi, poi il guadagno si attenua.

## B.4 Training set e test set

In [ ]:
X_B = ticket.drop(columns="ore_risoluzione")
y_B = ticket["ore_risoluzione"]

X_B_train, X_B_test, y_B_train, y_B_test = train_test_split(
    X_B, y_B, test_size=0.2, random_state=SEED
)
num_cols_B = ["priorita", "servizi_coinvolti", "riproducibile", "commenti_iniziali", "esperienza_assegnatario"]
cat_cols_B = ["componente"]
print("Training:", X_B_train.shape, " Test:", X_B_test.shape)

## B.5 Metriche e modelli di riferimento

Metriche di regressione:
- **MAE** (*Mean Absolute Error*): errore medio in valore assoluto, in ore. Il più facile da comunicare: "sbagliamo in media di X ore".
- **MSE** (*Mean Squared Error*): media degli errori al quadrato; penalizza molto gli errori grandi.
- **RMSE** (*Root MSE*): radice dell'MSE, di nuovo in ore.
- **R²** (coefficiente di determinazione): quota della variabilità del target spiegata dal modello; 1 = perfetto, 0 = come prevedere sempre la media.

Come riferimenti usiamo:
- `DummyRegressor`: prevede sempre la media delle ore del training set;
- `LinearRegression`: la **regressione lineare**, che assume effetti additivi e lineari.

In [ ]:
def metriche_reg(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {"MAE": mean_absolute_error(y_true, y_pred), "MSE": mse,
            "RMSE": np.sqrt(mse), "R2": r2_score(y_true, y_pred)}

def preproc_B():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols_B),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols_B),
    ])

risultati_B = {}
risultati_B["baseline (media)"] = metriche_reg(
    y_B_test, DummyRegressor().fit(X_B_train, y_B_train).predict(X_B_test))

lin_B = Pipeline([("prep", preproc_B()), ("lin", LinearRegression())]).fit(X_B_train, y_B_train)
risultati_B["regressione lineare"] = metriche_reg(y_B_test, lin_B.predict(X_B_test))
pd.DataFrame(risultati_B).T

## B.6 kNN: as-is, poi idiomatico

Stesso schema della Parte A. Qui le scale sono meno estreme (non ci sono migliaia di righe), ma `esperienza_assegnatario` e `commenti_iniziali` hanno comunque intervalli molto più ampi di `riproducibile` (0/1), che invece è una delle variabili più importanti.

In [ ]:
# --- AS-IS: nessuna standardizzazione ---
XtrB_raw = pd.get_dummies(X_B_train, columns=cat_cols_B, dtype=int)
XteB_raw = pd.get_dummies(X_B_test, columns=cat_cols_B, dtype=int).reindex(columns=XtrB_raw.columns, fill_value=0)

knn_raw_B = KNeighborsRegressor(n_neighbors=5).fit(XtrB_raw, y_B_train)
risultati_B["kNN as-is (k=5, no scaling)"] = metriche_reg(y_B_test, knn_raw_B.predict(XteB_raw))

# --- IDIOMATICO: Pipeline con standardizzazione ---
pipe_B = Pipeline([("prep", preproc_B()), ("knn", KNeighborsRegressor(n_neighbors=5))])
pipe_B.fit(X_B_train, y_B_train)
risultati_B["kNN idiomatico (k=5, scaling)"] = metriche_reg(y_B_test, pipe_B.predict(X_B_test))

pd.DataFrame(risultati_B).T

## B.7 Scelta di k e dei pesi con la cross-validazione

In regressione `GridSearchCV` massimizza un punteggio; per questo usiamo `neg_root_mean_squared_error`, cioè l'RMSE **cambiato di segno** (massimizzare −RMSE equivale a minimizzare l'RMSE).

In [ ]:
griglia_B = {
    "knn__n_neighbors": list(range(1, 41)),
    "knn__weights": ["uniform", "distance"],
}
cv_B = KFold(n_splits=5, shuffle=True, random_state=SEED)
gs_B = GridSearchCV(pipe_B, griglia_B, cv=cv_B, scoring="neg_root_mean_squared_error", n_jobs=-1)
gs_B.fit(X_B_train, y_B_train)

print("Migliori iperparametri:", gs_B.best_params_)
print(f"RMSE medio in cross-validazione: {-gs_B.best_score_:.2f} ore")

cvres_B = pd.DataFrame(gs_B.cv_results_)
for w, col in [("uniform", "#1b9e77"), ("distance", "#d95f02")]:
    sub = cvres_B[cvres_B["param_knn__weights"] == w]
    plt.plot(sub["param_knn__n_neighbors"], -sub["mean_test_score"], marker="o", ms=3, label=w, color=col)
plt.xlabel("k (numero di vicini)"); plt.ylabel("RMSE medio (CV, ore)")
plt.title("Scelta di k per la stima delle ore dei ticket")
plt.legend(title="weights"); plt.grid(alpha=0.3); plt.show()

## B.8 Valutazione finale sul test set

In [ ]:
best_B = gs_B.best_estimator_
y_B_pred = best_B.predict(X_B_test)
risultati_B[f"kNN ottimizzato {gs_B.best_params_}"] = metriche_reg(y_B_test, y_B_pred)

display(pd.DataFrame(risultati_B).T)

lim = [0, max(y_B_test.max(), y_B_pred.max()) * 1.05]
plt.scatter(y_B_test, y_B_pred, s=12, alpha=0.5, color="#7570b3")
plt.plot(lim, lim, "--", color="gray", label="previsione perfetta")
plt.xlim(lim); plt.ylim(lim)
plt.xlabel("ore reali"); plt.ylabel("ore previste")
plt.title("kNN ottimizzato: previsto vs reale (test set)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

**Lezione pratica n. 2**: sui dati con interazioni e non linearità il kNN (standardizzato e con k scelto bene) fa meglio della regressione lineare **senza** che abbiamo dovuto scrivere a mano i termini di interazione. In compenso la regressione lineare ci darebbe coefficienti interpretabili variabile per variabile: ogni modello ha il suo tipo di spiegabilità.

## B.9 Non solo un numero: una *forchetta* ricavata dai vicini

Per la pianificazione una stima puntuale ("12 ore") è meno utile di una **forchetta** ("tra 8 e 17 ore"). Con il kNN la otteniamo gratis guardando la dispersione delle ore dei vicini: minimo, massimo e quartili dei ticket simili.

In [ ]:
nuovo_ticket = pd.DataFrame([{
    "priorita": 2,
    "componente": "database",
    "servizi_coinvolti": 2,
    "riproducibile": 0,
    "commenti_iniziali": 4,
    "esperienza_assegnatario": 3.0,
}])

stima = best_B.predict(nuovo_ticket)[0]
k_B = max(best_B.named_steps["knn"].n_neighbors, 5)   # almeno 5 vicini per una forchetta sensata
vicini_B = mostra_vicini(best_B, X_B_train, y_B_train, nuovo_ticket, n=k_B)

q1, q3 = vicini_B["target"].quantile([0.25, 0.75])
print(f"Stima puntuale: {stima:.1f} ore")
print(f"Forchetta dai vicini (1° - 3° quartile): {q1:.1f} - {q3:.1f} ore")
vicini_B.rename(columns={"target": "ore_risoluzione"})

---
# Conclusioni: quando usare il kNN (e quando no)

| Punti di forza | Limiti |
|---|---|
| Spiegabile: ogni previsione è motivata da casi concreti | **Va sempre standardizzato**: è sensibile alle scale delle feature |
| Coglie non linearità e interazioni senza progettarle | Previsione lenta su dataset molto grandi: per ogni caso nuovo calcola distanze verso tutto il training set |
| Nessun vero addestramento: aggiungere nuovi casi è immediato | Soffre la **maledizione della dimensionalità**: con molte feature le distanze diventano tutte simili e perdono significato |
| Stesso codice per classificazione e regressione | Feature irrilevanti "inquinano" la distanza: conviene selezionarle |

### Collegamenti con il lavoro quotidiano
- **Rilevamento di ticket duplicati** e **suggerimento del revisore**: kNN su rappresentazioni del testo o del codice.
- **Ricerca semantica / RAG**: un database vettoriale non fa altro che un kNN (spesso **approssimato**, *ANN — Approximate Nearest Neighbors*, per essere veloce su milioni di vettori) sugli embedding.
- Per dataset grandi, `KNeighbors*` accetta `algorithm="kd_tree"` o `"ball_tree"`: strutture dati ad albero che evitano di confrontare il caso nuovo con tutti gli esempi.

### Esercizi proposti
1. Nella Parte A, rimuoviamo `commit_autore_nel_repo` e osserviamo come cambiano F1 e vicini.
2. Nella Parte B, proviamo `metric="manhattan"` (somma delle differenze in valore assoluto) nella griglia.
3. Aggiungiamo al dataset delle build 5 colonne di puro rumore casuale e verifichiamo quanto peggiora il kNN: è la maledizione della dimensionalità in azione.